# Build Latin Token Tables

Latin sibling of `Build Token Tables.ipynb` — same pipeline, applied to a small starting set of Latin hexameter epics known to have clean `book.line` locus structure. Deliberately a separate, duplicated notebook rather than a shared abstraction: the Latin side of this is still exploratory (data requirements — e.g. adding dependency-parse features — are still being worked out), so the mechanics are left exposed here to read, run, and modify directly, rather than hidden behind a library function. See `Build Token Tables.ipynb` for the Greek version and more detail on each step.

1. **Gets texts** from the Perseus Digital Library GitHub repo (`canonical-latinLit`).
2. **Extracts verse lines** from the XML, cleaning out editorial notes and deletions.
3. **Parses each text with spaCy** (`la_core_web_trf`) to produce token-level lemmatization and morphological analysis.
4. **Annotates speech and narration** using the DICES database of direct speech in Latin epic.
5. **Exports** token tables for each text as CSV (`data/tokens/{workgroup}.{work}.{edition}.csv`) where each row is a token with its verse location, morphology, and speech context.

## Preliminaries

### Install uva_common and dices-client

In [ ]:
# uva_common (with the nlp extra, for spaCy) and dices-client aren't
# preinstalled on a fresh runtime (e.g. Colab) the way they are in this
# repo's own local dev venv -- install explicitly so this notebook is
# runnable standalone
%pip install -q "uva_common[nlp] @ git+https://github.com/cwf2/uva_common"
%pip install -q git+https://github.com/cwf2/dices-client

### Install language model

In [ ]:
# Transformer-based Latin model — only install if not already present:
# pip can't skip the download based on the wheel URL alone (tested —
# even a "name @ url" requirement still re-fetches the full wheel), so
# check for the importable package first instead of relying on pip's
# own resolution
try:
    import la_core_web_trf
except ImportError:
    %pip install https://huggingface.co/latincy/la_core_web_trf/resolve/main/la_core_web_trf-3.9.6-py3-none-any.whl

### Import statements

Key dependencies:
- **`dicesapi`** — Client for the [DICES](https://dices.mta.ca/) API, used to identify direct speech in the texts.
- **`spacy`** — NLP pipeline for tokenization, lemmatization, and morphological parsing of Latin.

In [ ]:
import os
import pandas as pd
from dicesapi import DicesAPI
import uva_common
from tqdm import tqdm

### Global values

The repository record includes a pinned commit hash to ensure reproducibility — rerunning this notebook will always use the same snapshot of the Perseus texts.

**Text selection**: a starting set of four Latin epics confirmed (2026-09-16) to have clean `<tei:l>`-based verse-line markup and real DICES speech coverage — not the full Latin epic canon. Deliberately kept as a plain list here rather than in `CONFIG["texts"]`: which texts a given run processes is a parameter of the run, not shared library config, and some other Latin texts are known to have hairier loci (irregular book/line structure) that this pipeline hasn't been tested against — add texts here one at a time as they're verified, rather than assuming the rest of the Perseus Latin corpus will just work.

In [ ]:
# Texts to parse — (English name, URN) pairs
texts_to_parse = [
    ("Aeneid", "urn:cts:latinLit:phi0690.phi003.perseus-lat2"),
    ("Metamorphoses", "urn:cts:latinLit:phi0959.phi006.perseus-lat2"),
    ("Thebaid", "urn:cts:latinLit:phi1020.phi001.perseus-lat2"),
    ("Argonautica", "urn:cts:latinLit:phi1035.phi001.perseus-lat2"),
]

## Get texts

### Create local clone of Perseus repository

Clone the Perseus GitHub repo if it doesn't already exist locally, then reset to the pinned commit. This ensures we're working with a known, stable version of the texts.

In [ ]:
uva_common.clone_repo("latin")

### Populate text objects from local TEI XML

In [ ]:
print("Loading XML...")

# start with an empty list
corpus = []

# iterate over texts
for name, urn in texts_to_parse:

    # create record
    text = uva_common.Text(urn)

    # attach the English name (not reliably derivable from Perseus metadata)
    text.name = name

    # add to corpus
    corpus.append(text)

# check results
print(f"Corpus contains {len(corpus)} texts.")

## Parse text with spaCy

### Models

We use [`la_core_web_trf`](https://huggingface.co/latincy/la_core_web_trf), a transformer-based spaCy model for Latin (LatinCy, Patrick Burns). It provides lemmatization, POS tagging, and morphological analysis.

### Parsing

Each text's lines are joined into a single string before parsing. This gives the transformer model full cross-line context, which improves accuracy for morphological disambiguation — especially important for a highly inflected language like Latin.

**Note:** timing not yet benchmarked for this corpus (see `Build Token Tables.ipynb` for the Greek pipeline's benchmark, on the same hardware, as a rough reference point).

### Token tables

The token table maps each spaCy token back to its verse line using the character-offset index built earlier, then extract morphological features into a flat table. Each row in the resulting DataFrame represents one token with columns for:
- **location** — CTS URN, author, work, book/prefix, line number, sequence
- **token data** — surface form and lemma
- **morphology** — POS, verb form, mood, tense, voice, person, number, case, gender

In [ ]:
# iterate over texts
for i, text in enumerate(corpus):
    print(f"[{i+1}/{len(corpus)}]", text.author, text.title, end="...")

    # run NLP pipeline
    tokens = text.parse()

    # check that it worked
    print(len(tokens), "tokens")

    # cache result on text object
    text.tokens = tokens

## Apply manual corrections

In [ ]:
# remove punctuation
for text in corpus:
    text.tokens = text.tokens.loc[~(text.tokens["pos"] == "PUNCT")].reset_index(drop=True)

No Latin-specific accent/elision corrections exist yet (the Greek pipeline's `accent_corrections.tsv`/`lemma_corrections.tsv` are derived from a Greek-only spreadsheet and wouldn't match any Latin lemma, so they're deliberately not applied here rather than run as a silent no-op). If similar data-quality issues turn up in the Latin lemmas, follow the same pattern as `generate_accent_corrections.py` to build a Latin equivalent.

## Identify speech/narration
### instantiate API

In [ ]:
api = DicesAPI()

### download full speech list

In [ ]:
all_speeches = api.getSpeeches()
print(len(all_speeches), "speeches")

### edit known speeches

This is necessary to fix line differences between editions used by DICES and Perseus. None known yet for this Latin text set — this cell is a no-op for now, kept as a template (see `Build Token Tables.ipynb` for two real examples from the Greek pipeline). If a speech's line range doesn't match up against the token table below (a "could not locate speech" warning), that's usually an edition mismatch to fix here.

In [ ]:
for s in all_speeches:
    # no known edition mismatches yet for this text set -- add fixes here
    # following the pattern in Build Token Tables.ipynb if/when one turns up
    pass

## Annotate and export per text

In [ ]:
tokens_dir = os.path.join(uva_common.CONFIG["data_dir"], "tokens")
os.makedirs(tokens_dir, exist_ok=True)

for text in corpus:
    print(text.author, text.title, "...", end=" ")

    # ordered sequence of line URNs taken directly from the token table
    line_table = pd.DataFrame({
        "urn": text.tokens["urn"].drop_duplicates()
    }).reset_index(drop=True)
    line_table = line_table.assign(
        speech_id=None, speaker=None, addressee=None,
        level=0, type=None, cluster=None, turn=None, tags=None,
    )

    # annotate lines that fall inside speeches
    for speech in all_speeches:
        if speech.work.urn != text.urn:
            continue

        first_urn = f"{text.urn}:{speech.l_fi}"
        last_urn = f"{text.urn}:{speech.l_la}"

        matches_first = line_table.index[line_table["urn"] == first_urn]
        matches_last  = line_table.index[line_table["urn"] == last_urn]

        if len(matches_first) == 0 or len(matches_last) == 0:
            print(f"\n  warning: could not locate speech {speech._attributes['public_id']} "
                  f"({speech.l_fi}\u2013{speech.l_la})")
            continue

        i_first = matches_first[0]
        i_last  = matches_last[0]

        line_table.loc[i_first:i_last, "speech_id"]  = speech._attributes["public_id"]
        line_table.loc[i_first:i_last, "speaker"]    = speech.getSpkrString()
        line_table.loc[i_first:i_last, "addressee"]  = speech.getAddrString()
        line_table.loc[i_first:i_last, "level"]      = speech.level + 1
        line_table.loc[i_first:i_last, "type"]       = speech.type
        line_table.loc[i_first:i_last, "cluster"]    = speech.cluster.id
        line_table.loc[i_first:i_last, "turn"]       = speech.part
        line_table.loc[i_first:i_last, "tags"]       = ";".join(
            tag["type"] for tag in speech._attributes["tags"]
        )

    # NOTE: the Greek pipeline distinguishes Odysseus' apologue (Odyssey
    # 9-12) from his other speeches by public_id here. No equivalent
    # embedded-narration special case is known yet for this Latin text set
    # -- add one the same way if one turns up.

    # merge speech annotation into token table
    text.tokens = pd.merge(text.tokens, line_table, on="urn")

    # attach the English work name as its own column
    text.tokens.insert(2, "work", text.name)

    # export — filename built from CTS components (colon-free, OS-independent),
    # not the URN string itself
    outfile = os.path.join(tokens_dir, f"{text.workgroup}.{text.work}.{text.edition}.csv")
    text.tokens.to_csv(outfile, index=False)
    print(f"{len(text.tokens)} tokens \u2192 {outfile}")

## Sanity checks
### speech and narrative token counts per text

In [ ]:
for text in corpus:
    n_total    = len(text.tokens)
    n_speech   = (text.tokens["level"] > 0).sum()
    n_narr     = (text.tokens["level"] == 0).sum()
    pct_speech = 100 * n_speech / n_total if n_total else 0
    print(f"{text.author} {text.title}: "
          f"{n_total} tokens, {n_speech} speech ({pct_speech:.1f}%), {n_narr} narrative")

### are any lines being counted twice?

In [ ]:
for text in corpus:
    dupes = (
        text.tokens
        .groupby("urn")["speech_id"]
        .nunique()
        .pipe(lambda s: s[s > 1])
    )
    if len(dupes):
        print(f"WARNING \u2014 {text.title}: {len(dupes)} URNs with conflicting speech_id")
        print(dupes)
    else:
        print(f"{text.title}: ok")